# Download CIFAR-10 into your Google Drive
Run these cells in Google Colab using your intended Google account. This downloads the official complete CIFAR-10 archive through torchvision, validates its checksums, extracts the original data, and checks the cat/dog counts. No training is started. No paid runtime is required.

Destination: `MyDrive/lost-or-inaccessible/datasets/`. Existing valid data are reused. Drive access must be authorized interactively by you. The completion marker is written only after verification succeeds.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT_STORAGE = Path('/content/drive/MyDrive/lost-or-inaccessible')
DATA_DIR = PROJECT_STORAGE / 'datasets'
RESULTS_DIR = PROJECT_STORAGE / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset destination:', DATA_DIR)


In [ ]:
import torchvision
from torchvision.datasets import CIFAR10
print('torchvision:', torchvision.__version__)
print('Official archive:', CIFAR10.url)
train = CIFAR10(root=str(DATA_DIR), train=True, download=True)
test = CIFAR10(root=str(DATA_DIR), train=False, download=True)
print('Official dataset downloaded and checked.')


In [ ]:
import hashlib, json
from datetime import datetime, timezone
from collections import Counter
train_counts = Counter(train.targets)
test_counts = Counter(test.targets)
assert len(train) == 50000 and len(test) == 10000
assert train_counts[3] == train_counts[5] == 5000
assert test_counts[3] == test_counts[5] == 1000
assert train._check_integrity() and test._check_integrity()
archive = DATA_DIR / CIFAR10.filename
archive_md5 = None
if archive.exists():
    digest = hashlib.md5()
    with archive.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    archive_md5 = digest.hexdigest()
    assert archive_md5 == CIFAR10.tgz_md5, 'Archive checksum mismatch'
status = {
    'status': 'verified',
    'verified_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': 'official CIFAR-10; original files unchanged',
    'source': CIFAR10.url,
    'directory': str(DATA_DIR),
    'official_train_examples': len(train),
    'official_test_examples': len(test),
    'cat_dog_train_examples': train_counts[3] + train_counts[5],
    'cat_dog_test_examples': test_counts[3] + test_counts[5],
    'archive_md5': archive_md5,
    'torchvision_file_checksums': 'passed',
    'research_experiments': 'not run',
}
marker = DATA_DIR / 'DATASET_VERIFIED.json'
marker.write_text(json.dumps(status, indent=2))
print(json.dumps(status, indent=2))
print('SUCCESS: dataset saved and verified in Google Drive.')


## Next: run the research project
Use `colab_full_pipeline.ipynb` with `USE_DRIVE = True`. It uses the same Drive dataset and results directories. Upload/extract the repository ZIP into the Colab runtime as described there. Training uses only cats/dogs, splits the official training examples reproducibly, and keeps the official test subset held out. This setup notebook does not train a model or produce research findings.